# Attune Benchmarks

This notebook is the single entry point for pitch, note, and mistake benchmarks. The benchmark code lives in `PitchBenchmarker.py`, `NoteBenchmarker.py`, and `MistakeBenchmarker.py`.

Etude MIDI files are read from `benchmarks/datasets/violin-etudes/<dataset>/midi/`. Production pitch detection is cached under each corpus' `pitch_data/` directory after a smoothed run, then reused by note and mistake benchmarks.

In [1]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from PitchBenchmarker import PitchBenchmarker
from NoteBenchmarker import NoteBenchmarker
from MistakeBenchmarker import MistakeBenchmarker, MistakeInjector
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

## Pitch Detection

Runs pYIN detector + smoother by default and writes/reads compressed `PitchData` caches.

In [7]:
pb = PitchBenchmarker()
pitch_df = pb.bench_pitch_dataset("mdb-stem-synth", max_tracks=4, write=True)
pb.summarize(pitch_df, cols=pb.PITCH_METRICS + ["pitch_detector_compute_time", "pitch_smoother_compute_time"], name="mdb-stem-synth")
pitch_df.head()

[mdb-stem-synth]   1/4 AClassicEducation_NightOwl_STEM_01.R RPA=0.793 OA=0.797
[mdb-stem-synth]   2/4 AClassicEducation_NightOwl_STEM_08.R RPA=0.823 OA=0.886
[mdb-stem-synth]   3/4 AClassicEducation_NightOwl_STEM_13.R RPA=0.822 OA=0.926
[mdb-stem-synth]   4/4 AimeeNorwich_Child_STEM_02.RESYN     RPA=0.799 OA=0.819

=== mdb-stem-synth: mean over 4 rows ===
Raw Pitch Accuracy             0.8094
Raw Chroma Accuracy            0.8133
Overall Accuracy               0.8568
Voicing Recall                 0.9788
Voicing False Alarm            0.0885
pitch_detector_compute_time   28.0816
pitch_smoother_compute_time   15.6825


,Voicing Recall,Voicing False Alarm,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,pitch_detector_compute_time,pitch_smoother_compute_time,pitch_compute_time,fmin,fmax
track_id,,,,,,,,,,
AClassicEducation_NightOwl_STEM_01.RESYN,0.9680,0.1658,0.7927,0.8048,0.7967,42.7660,21.3781,64.1441,39.1878,182.2501
AClassicEducation_NightOwl_STEM_08.RESYN,0.9849,0.0549,0.8235,0.8247,0.8859,17.8684,14.0308,31.8992,118.3036,405.8544
AClassicEducation_NightOwl_STEM_13.RESYN,0.9837,0.0296,0.8224,0.8224,0.9261,11.4395,11.5458,22.9853,113.6719,343.9052
AimeeNorwich_Child_STEM_02.RESYN,0.9787,0.1037,0.7991,0.8013,0.8187,40.2528,15.7751,56.0279,47.5097,164.4865


## Note Detection

Synthesizes etude MIDI, loads or creates production pitch caches, then times each note detector method separately.

In [ ]:
from pathlib import Path

nb = NoteBenchmarker(onset_tolerance=0.05)
kayser_df = nb.bench_note_dataset("kayser", max_tracks=2, verbose=True, write=True)
print(f"wrote {nb.result_csv_path('note', 'kayser')}")

summary_cols = ["Precision", "Recall", "F-measure", "Average Overlap Ratio", "note_compute_time"]
kayser_summary = (
    kayser_df.groupby(["method", "refined_with_onsets", "transition_excluding"], dropna=False)[summary_cols]
    .mean(numeric_only=True)
    .sort_values("F-measure", ascending=False)
)
display(kayser_summary)
display(kayser_df[kayser_df["error"].notna()][["track", "method", "error"]])


### Basic Pitch Backend Diagnostics

Run this when the `basic-pitch` benchmark row reports a local backend/model-load error. It prints installed backend modules, the packaged model path, then tries one direct `basic_pitch.inference.predict()` call on the first Kayser synth WAV.

In [ ]:
from pathlib import Path
import importlib
import importlib.metadata as importlib_metadata
import os
import traceback

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "attune-matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

backend_modules = ["basic_pitch", "tensorflow", "coremltools", "tflite_runtime", "onnxruntime"]
for module_name in backend_modules:
    spec = importlib.util.find_spec(module_name)
    version = None
    package_name = module_name.replace("_", "-")
    try:
        version = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        pass
    print(f"{module_name:15s} installed={spec is not None} version={version} origin={getattr(spec, 'origin', None)}")

try:
    import basic_pitch
    from basic_pitch import ICASSP_2022_MODEL_PATH
    from basic_pitch.inference import predict

    model_path = Path(ICASSP_2022_MODEL_PATH)
    print("basic_pitch package:", Path(basic_pitch.__file__).resolve())
    print("model path:", model_path)
    print("model exists:", model_path.exists(), "is_dir:", model_path.is_dir())
    if model_path.exists() and model_path.is_dir():
        print("model files:", sorted(p.name for p in model_path.iterdir())[:20])

    pb = PitchBenchmarker()
    midi_path = next((pb.DATASETS / "violin-etudes" / "kayser" / "midi").glob("*.mid"))
    wav_path = pb.synth_midi(midi_path)
    print("test wav:", wav_path)

    model_output, midi_data, note_events = predict(
        str(wav_path),
        ICASSP_2022_MODEL_PATH,
        minimum_frequency=196.0,
        maximum_frequency=3000.0,
    )
    print("Basic Pitch succeeded; note events:", len(note_events))
    display(note_events[:10])
except Exception:
    traceback.print_exc()


## Mistake Detection

Symbolic mode isolates MistakeDetector. Audio mode renders injected performances, loads or creates production pitch caches, then reports note/mistake-detection/mistake-check timing.

In [ ]:
mb = MistakeBenchmarker()
injector = MistakeInjector(mistake_rate=0.2, weights=(0.4, 0.3, 0.3))
symbolic = mb.bench_mistake_dataset("kayser", injector, seeds=range(3), mode="symbolic", max_tracks=3, verbose=False, write=True)
symbolic

## Full Pitch Run

Optional heavier run across all pitch corpora.

In [ ]:
full_pitch = {}
for ds in pb.PITCH_DATASETS:
    df = pb.bench_pitch_dataset(ds, verbose=False, write=True)
    full_pitch[ds] = df[pb.PITCH_METRICS + ["pitch_detector_compute_time", "pitch_smoother_compute_time"]].mean(numeric_only=True)
pd.DataFrame(full_pitch).T